# Logistic Regression & Calibration — AI Lab Instructor Notebook

*Classical ML · Medium*

Treat probabilistic outputs as first-class: calibration, class imbalance, thresholding, and metric tradeoffs.

**Outcome.** Students can implement logistic regression predictions, compute calibration diagnostics (reliability curve, ECE), and reason about imbalanced evaluation (PR vs ROC) like a FAANG ML engineer.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Logistic Regression & Calibration — Student Lab

We focus on *probabilities*, not just accuracy.

## Section 0 — Synthetic imbalanced data
We simulate logits and labels with imbalance and miscalibration.

### Figure: make_probs data overview
<div style="max-width: 70%;"><img src="/images/w3-d3-make-probs-overview.svg" alt="make_probs synthetic data overview" /></div>

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_probs(n=5000, base_rate=0.05, logit_scale=1.0, miscalibration=1.0):
    z = logit_scale * rng.standard_normal(n)
    z = z + np.log(base_rate/(1-base_rate))
    p_true = 1/(1+np.exp(-z))
    y = (rng.random(n) < p_true).astype(int)
    z_model = miscalibration * z
    p_model = 1/(1+np.exp(-z_model))
    return y, p_model, p_true

y, p_model, p_true = make_probs(miscalibration=2.0)
print('base_rate', y.mean())
check('in_0_1', np.all((p_model>=0) & (p_model<=1)))

**Why this works.** # Logistic Regression & Calibration — Instructor (Solutions + Rationale)

## Task 2: Confusion matrix metrics at a threshold
*Section: Metrics*

## Section 1 — Metrics

### Task 1.1: Confusion matrix metrics at a threshold
Implement precision, recall, F1 at threshold t.

- y_hat = (p>=t)
- TP/FP/FN

**Checkpoint:** Why is accuracy misleading under imbalance?

<div style="max-width: 70%;"><img src="/images/w3-d3-precision-recall-tradeoff.svg" alt="Precision vs recall tradeoff" /></div>

**Hints**
- y_hat = (p>=t)
TP/FP/FN

In [ ]:
def metrics_at_threshold(y, p, t):
    y = y.astype(int)
    yhat = (p >= t).astype(int)
    tp = int(np.sum((yhat==1) & (y==1)))
    fp = int(np.sum((yhat==1) & (y==0)))
    fn = int(np.sum((yhat==0) & (y==1)))
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2*prec*rec / (prec+rec+1e-12)
    return {'tp':tp,'fp':fp,'fn':fn,'precision':prec,'recall':rec,'f1':f1}

print(metrics_at_threshold(y, p_model, 0.5))
# Rationale: for imbalance, precision/recall capture tradeoff better than accuracy.

**Why this works.** for imbalance, precision/recall capture tradeoff better than accuracy.

## Task 3: PR curve area (approx)
*Section: Metrics*

### Task 1.2: PR curve area (approx)
Compute a simple PR-AUC approximation by sorting thresholds.

- sort by p desc
- compute precision/recall at each cut

**Interview Angle:** when is PR-AUC preferable to ROC-AUC?

### Figure: PR-AUC and ROC-AUC curves
<div style="max-width: 70%;"><img src="/images/w3-d3-pr-auc-curve.svg" alt="PR-AUC curve" /></div>
<div style="max-width: 70%;"><img src="/images/w3-d3-roc-auc-curve.svg" alt="ROC-AUC curve" /></div>

**Hints**
- sort by p desc
compute precision/recall at each cut

In [ ]:
def pr_curve(y, p):
    order = np.argsort(-p)
    y_sorted = y[order]
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp[-1] + 1e-12)
    return rec, prec

def auc_trapz(x, y):
    # assume x increasing
    return float(np.trapz(y, x))

rec, prec = pr_curve(y, p_model)
pr_auc = auc_trapz(rec, prec)
print('pr_auc', pr_auc)
check('finite', np.isfinite(pr_auc))
# Rationale: PR is more sensitive to minority class performance.

**Why this works.** PR is more sensitive to minority class performance.

## Task 4: Reliability curve + ECE
*Section: Calibration*

## Section 2 — Calibration

### Task 2.1: Reliability curve + ECE

Bin probabilities and compute:
- avg predicted prob per bin
- empirical accuracy per bin
- ECE = sum (bin_weight * |acc - conf|)

- np.digitize

**FAANG gotcha:** model can have good ranking but bad calibration.

### Figure: Reliability curve
<div style="max-width: 70%;"><img src="/images/w3-d3-reliability-curve.svg" alt="Reliability curve before and after temperature scaling" /></div>

### Calibration intuition table

| Well-calibrated (Predicted → Actual) | Poorly calibrated (Predicted → Actual) |
|---|---|
| 0.1 → ~10% | 0.1 → ~25% (underconfident) |
| 0.5 → ~50% | 0.5 → ~50% |
| 0.9 → ~90% | 0.9 → ~70% (overconfident) |

**Hints**
- np.digitize

In [ ]:
def reliability_bins(y, p, n_bins=10):
    y = y.astype(int)
    edges = np.linspace(0, 1, n_bins+1)
    b = np.digitize(p, edges[1:-1], right=False)
    bin_acc = np.zeros(n_bins)
    bin_conf = np.zeros(n_bins)
    bin_frac = np.zeros(n_bins)
    for i in range(n_bins):
        mask = (b == i)
        if mask.any():
            bin_acc[i] = y[mask].mean()
            bin_conf[i] = p[mask].mean()
            bin_frac[i] = mask.mean()
    return bin_acc, bin_conf, bin_frac

def ece(bin_acc, bin_conf, bin_frac):
    return float(np.sum(bin_frac * np.abs(bin_acc - bin_conf)))

bin_acc, bin_conf, bin_frac = reliability_bins(y, p_model, n_bins=10)
ECE = ece(bin_acc, bin_conf, bin_frac)
print('ECE', ECE)
check('ece_finite', np.isfinite(ECE))
# Rationale: ECE measures calibration gap; ranking can be good even if calibration is poor.

**Why this works.** ECE measures calibration gap; ranking can be good even if calibration is poor.

## Task 5: Temperature scaling
*Section: Calibration*

### Task 2.2: Temperature scaling

We assume p_model came from logits z_model. Approximate logits via logit(p).
Then find temperature T that minimizes NLL on validation split: sigmoid(z/T).

- logit(p)=log(p/(1-p))
- grid search T over [0.5..5]

**Checkpoint:** Why does scaling logits preserve ranking?

**Temperature scaling Intuition (no math)**

Temperature scaling fixes confidence, not correctness.

It makes probabilities honest without changing predictions or ranking. It does NOT work on probabilities directly. It works on logits.

Think of logits as volume:

Your model is shouting: "THIS IS 95%!!!"

Reality is calmer: "It’s more like 70%."

Temperature scaling just says: "Lower the volume."

**Hints**
- logit(p)=log(p/(1-p))
grid search T over [0.5..5]

In [ ]:
def logit(p, eps=1e-12):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def nll(y, p, eps=1e-12):
    p = np.clip(p, eps, 1-eps)
    return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))

idx = rng.permutation(len(y))
val = idx[: len(y)//2]
test = idx[len(y)//2:]

z = logit(p_model)

Ts = np.linspace(0.5, 5.0, 50)
best_T = None
best_loss = float('inf')
for T in Ts:
    pT = 1/(1+np.exp(-(z[val]/T)))
    L = nll(y[val], pT)
    if L < best_loss:
        best_loss = L
        best_T = T

p_cal = 1/(1+np.exp(-(z[test]/best_T)))
ECE_before = ece(*reliability_bins(y[test], p_model[test], 10))
ECE_after = ece(*reliability_bins(y[test], p_cal, 10))
print('best_T', best_T)
print('ECE_before', ECE_before, 'ECE_after', ECE_after)
# Rationale: temperature scaling rescales logits, improving calibration while preserving ranking.

**Why this works.** temperature scaling rescales logits, improving calibration while preserving ranking.

## Task 6: Pick threshold minimizing cost
*Section: Thresholding with costs*

## Section 3 — Thresholding with costs

### Task 3.1: Pick threshold minimizing cost
Cost = c_fp*FP + c_fn*FN


**Interview Angle:** map model probabilities to business decisions.

### Figure: Cost vs threshold
<div style="max-width: 70%;"><img src="/images/w3-d3-threshold-cost-curve.svg" alt="Threshold vs business cost curve" /></div>

In [ ]:
def best_threshold_cost(y, p, c_fp=1.0, c_fn=10.0):
    ts = np.linspace(0, 1, 501)
    best_t = 0.5
    best_cost = float('inf')
    for t in ts:
        yhat = (p >= t).astype(int)
        fp = np.sum((yhat==1) & (y==0))
        fn = np.sum((yhat==0) & (y==1))
        cost = c_fp*fp + c_fn*fn
        if cost < best_cost:
            best_cost = cost
            best_t = float(t)
    return best_t, float(best_cost)

t_star, cost_star = best_threshold_cost(y, p_model, c_fp=1.0, c_fn=10.0)
print('t*', t_star, 'cost', cost_star)
# Rationale: thresholding is a decision problem; optimize costs, not accuracy.

**Why this works.** thresholding is a decision problem; optimize costs, not accuracy.